In [17]:
import pandas as pd
import pickle
import numpy as np

In [ ]:
# ==============================================================================
# 1. GLOBAL MODEL INITIALIZATION
# ==============================================================================
# Objective: Load the serialized machine learning artifacts from the disk.
# 
# The 'model_progress.pickle' file is a dictionary container that holds:
# - 'models_dict': A collection of trained models for each target (Weight, BodyFat, etc.).
# - 'encoders': The Scikit-Learn LabelEncoders used to translate text to numbers.
# - 'features': The strict list of input columns required for accurate prediction.

# 1. Load Model
# Open the file in binary read mode ('rb') to deserialize the data structure.
with open('../../models/model_progress.pickle', 'rb') as f:
    data_model = pickle.load(f)

# Extract components into global variables for easy access during inference.
models = data_model['models_dict']
encoders = data_model['encoders']
feature_order = data_model['features']

In [ ]:
# ==============================================================================
# 4. PREDICTION ENGINE (WEEKLY PROGRESS)
# ==============================================================================

def prediksi_progres_lengkap(user_data, minggu_ke):
    """
    Predicts the user's physical status and nutritional needs for a specific week.

    This function utilizes a collection of pre-trained Machine Learning models 
    to forecast various metrics (Weight, Body Fat, Calories, Macros) based on 
    the user's initial profile and the elapsed time (week number).

    ---------------------------------------------------------------------------
    Args:
        user_data (dict): A dictionary containing the user's initial profile.
                          Required Keys:
                          - 'Age' (int), 'Gender' (str), 'Height_cm' (float)
                          - 'Initial_Weight_kg' (float)
                          - 'Goal' (str: 'Muscle Gain'/'Weight Loss')
                          - 'level' (str: 'Beginner'/'Intermediate'/'Advanced')
                          - 'Body_Fat_Percentage_x' (float)
                          - 'Workout_Frequency' (int)
                          - 'Average_Duration_Minutes' (int)
                          - Sports Flags (0/1): 'Badminton', 'Football', etc.

        minggu_ke (int): The target week number for prediction (e.g., 1, 4, 12).
                         This serves as the primary time-series variable for the models.

    Returns:
        dict: A dictionary containing all predicted metrics for that specific week.
              Example:
              {
                  'Weight_kg': 60.5,
                  'BMI': 21.0,
                  'Daily_Calories': 2500,
                  'Target_Protein_g': 150,
                  ...
              }

    ---------------------------------------------------------------------------
    Operational Logic:
    
    A. Feature Engineering (Derived Data):
       Calculates the initial BMI and determines the BMI Category (Underweight/
       Normal/Obese) automatically to reduce user input friction.

    B. Data Encoding (Input Conversion):
       Transforms categorical text data (Gender, Goal, Level) into numerical 
       formats using pre-trained LabelEncoders, making them compatible with 
       the ML models.

    C. Multi-Model Prediction (Parallel Inference):
       Iterates through the 'models' dictionary. Each target metric (e.g., Weight, 
       Calories) has its own dedicated regression or classification model that 
       predicts independently based on the same input vector.

    D. Output Decoding (Human-Readable Translation):
       - Numerical outputs (e.g., Weight) are stored directly.
       - Categorical outputs (e.g., BMI Status) are inversely transformed from 
         numbers back to text strings for readability.
    """

    # --- A. FEATURE ENGINEERING (Calculate Derived Metrics) ---
    tinggi_m = user_data['Height_cm'] / 100
    bmi_awal = round(user_data['Initial_Weight_kg'] / (tinggi_m ** 2), 2)
    
    # Determine Initial BMI Category
    if bmi_awal < 18.5: cat_bmi = 'Underweight'
    elif bmi_awal < 25: cat_bmi = 'Normal'
    elif bmi_awal < 30: cat_bmi = 'Overweight'
    else: cat_bmi = 'Obese'
    
    # --- B. DATA ENCODING (Text -> Number) ---
    # Using pre-trained encoders to ensure consistency with training data
    gender_code = encoders['Gender'].transform([user_data['Gender']])[0]
    goal_code = encoders['Goal'].transform([user_data['Goal']])[0]
    level_code = encoders['level'].transform([user_data['level']])[0]
    bmi_cat_code = encoders['BMI_Category_x'].transform([cat_bmi])[0]

    # --- C. CONSTRUCT INPUT ARRAY ---
    # CRITICAL: The order of features here MUST match 'feature_order' exactly.
    input_row = [
        user_data['Age'], 
        gender_code, 
        user_data['Height_cm'], 
        user_data['Initial_Weight_kg'],
        bmi_awal, 
        bmi_cat_code, 
        user_data.get('Body_Fat_Category', 0), # Use .get() for safety
        user_data['Body_Fat_Percentage_x'],
        goal_code, 
        user_data['Workout_Frequency'], 
        user_data['Average_Duration_Minutes'], 
        level_code,
        user_data.get('Badminton', 0), 
        user_data.get('Football', 0), 
        user_data.get('Basketball', 0),
        user_data.get('Volleyball', 0), 
        user_data.get('Swim', 0),
        minggu_ke # Time variable
    ]
    
    # Wrap in DataFrame for compatibility with model.predict()
    input_df = pd.DataFrame([input_row], columns=feature_order)
    
    # --- D. PREDICTION LOOP (ALL MODELS) ---
    hasil = {}
    
    # Iterate through every loaded model in the dictionary
    # target_name: Column name (e.g., 'Weight_kg', 'Progress_Status_Encoded')
    # info: Dictionary containing {'model': model_object, 'type': 'numeric'/'categorical'}
    for target_name, info in models.items():
        
        # 1. Predict Value
        nilai_prediksi = info['model'].predict(input_df)[0]
        
        # 2. Handle Output Type
        if info['type'] == 'categorical':
            # DECODING: Convert predicted number back to text
            # The encoder name is usually the target name without "_Encoded" suffix
            nama_asli = target_name.replace('_Encoded', '')
            
            # Inverse Transform
            try:
                teks = encoders[nama_asli].inverse_transform([int(nilai_prediksi)])[0]
                hasil[nama_asli] = teks
            except:
                hasil[nama_asli] = "Unknown" # Fallback if decoding fails
        else:
            # NUMERIC: Store value directly
            hasil[target_name] = nilai_prediksi
            
    return hasil

In [ ]:
# ==============================================================================
# 5. SIMULATION & TESTING (EXECUTION)
# ==============================================================================
# Objective: Validate the model's logic by simulating a 12-week user journey.
#
# This section defines a "Mock User" profile and iterates through a 12-week 
# timeline. It calls the prediction engine for each week to observe how the 
# physical stats (Weight, BMI) and nutritional needs adapt over time.

# 1. Define Mock User Data (Test Case)
# Represents the initial state (T=0) of a user starting their fitness journey.
input_user = {
    'Age': 25, 
    'Gender': 'Male', 
    'Height_cm': 175, 
    'Initial_Weight_kg': 60,
    'Body_Fat_Category': 2, 
    'Body_Fat_Percentage_x': 15.0,
    'Goal': 'Muscle Gain', 
    'Workout_Frequency': 4, 
    'Average_Duration_Minutes': 60,
    'level': 'Beginner',
    # Activity Flags (Binary 0/1)
    'Badminton': 0, 
    'Football': 1, 
    'Basketball': 0, 
    'Volleyball': 0, 
    'Swim': 0
}

print(f"User: Pria, 25th, 60kg -> Goal: Muscle Gain")

# 2. Execution Loop (Weeks 1 to 12)
# Iterates through the timeline to generate a trajectory of progress.
for minggu in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]:
    
    # Invoke Prediction Engine
    output = prediksi_progres_lengkap(input_user, minggu_ke=minggu)
    
    print(f"\n{'='*10} MINGGU KE-{minggu} {'='*10}")
    
    # Output Grouping for Readability
    # A. Physical Metrics
    print(f"[FISIK]")
    print(f"  Berat Badan     : {output['Weight_kg']:.2f} kg")
    print(f"  BMI             : {output['BMI']:.2f} ({output['BMI_Category_y']})")
    print(f"  Body Fat        : {output['Body_Fat_Percentage_y']:.1f}%")
    
    # B. Daily Nutrition Targets
    print(f"[NUTRISI HARIAN]")
    print(f"  Kalori          : {output['Daily_Calories']:.0f} kkal")
    print(f"  Air Minum       : {output['Daily_Water_ml']:.0f} ml")
    print(f"  Gula (Limit)    : {output['Limit_Sugar_g']:.1f} g")
    
    # C. Macronutrient Distribution
    print(f"[MAKRO NUTRISI]")
    print(f"  Protein         : {output['Target_Protein_g']:.1f} g")
    print(f"  Karbo           : {output['Target_Carbs_g']:.1f} g")
    print(f"  Lemak           : {output['Target_Fat_g']:.1f} g")
    print(f"  Serat           : {output['Target_Fiber_g']:.1f} g")
    print(f"  Kalsium         : {output['Target_Calcium_mg']:.0f} mg")
    print(f"  Kolesterol (Max): {output['Limit_Cholesterol_mg']:.0f} mg")

User: Pria, 25th, 60kg -> Goal: Muscle Gain

========== MINGGU KE-1 ==========
[FISIK]
  Berat Badan     : 60.23 kg
  BMI             : 19.46 (Normal)
  Body Fat        : 14.9%
[NUTRISI HARIAN]
  Kalori          : 2703 kkal
  Air Minum       : 2741 ml
  Gula (Limit)    : 67.0 g
[MAKRO NUTRISI]
  Protein         : 204.3 g
  Karbo           : 326.8 g
  Lemak           : 58.4 g
  Serat           : 37.1 g
  Kalsium         : 1000 mg
  Kolesterol (Max): 300 mg

========== MINGGU KE-2 ==========
[FISIK]
  Berat Badan     : 60.26 kg
  BMI             : 19.51 (Normal)
  Body Fat        : 14.8%
[NUTRISI HARIAN]
  Kalori          : 2703 kkal
  Air Minum       : 2741 ml
  Gula (Limit)    : 67.0 g
[MAKRO NUTRISI]
  Protein         : 204.3 g
  Karbo           : 326.8 g
  Lemak           : 58.4 g
  Serat           : 37.1 g
  Kalsium         : 1000 mg
  Kolesterol (Max): 300 mg

========== MINGGU KE-3 ==========
[FISIK]
  Berat Badan     : 60.30 kg
  BMI             : 19.54 (Normal)
  Body Fat       


========== MINGGU KE-4 ==========
[FISIK]
  Berat Badan     : 60.42 kg
  BMI             : 19.58 (Normal)
  Body Fat        : 14.6%
[NUTRISI HARIAN]
  Kalori          : 2706 kkal
  Air Minum       : 2752 ml
  Gula (Limit)    : 67.1 g
[MAKRO NUTRISI]
  Protein         : 204.5 g
  Karbo           : 327.1 g
  Lemak           : 58.5 g
  Serat           : 37.1 g
  Kalsium         : 1000 mg
  Kolesterol (Max): 300 mg

========== MINGGU KE-5 ==========
[FISIK]
  Berat Badan     : 60.54 kg
  BMI             : 19.61 (Normal)
  Body Fat        : 14.6%
[NUTRISI HARIAN]
  Kalori          : 2706 kkal
  Air Minum       : 2752 ml
  Gula (Limit)    : 67.1 g
[MAKRO NUTRISI]
  Protein         : 204.5 g
  Karbo           : 327.1 g
  Lemak           : 58.5 g
  Serat           : 37.1 g
  Kalsium         : 1000 mg
  Kolesterol (Max): 300 mg

========== MINGGU KE-6 ==========
[FISIK]
  Berat Badan     : 60.62 kg
  BMI             : 19.65 (Normal)
  Body Fat        : 14.5%
[NUTRISI HARIAN]
  Kalori          